In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

import pandas as pd
import numpy as np

from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
results_dir = Path("/Users/sylvi/topo_data/dna_damage_cache/analysis_results")
results_csv_file = results_dir / "defect_grain_statistics.csv"
assert results_csv_file.exists(), f"Results CSV file does not exist: {results_csv_file}"
df = pd.read_csv(results_csv_file)
print(df.columns)
original_df = df.copy()

folder_groups = [
    [["TE/5_percent_damage", "TE/20_percent_damage", "TE/50_percent_damage"], "tomato"],
    [["MilliQ/5_percent_damage", "MilliQ/20_percent_damage", "MilliQ/50_percent_damage"], "cornflowerblue"],
    [["Controls"], "lightgray"],
]

In [ ]:
cols_to_pca = [
    "num_height_defects",
    "num_curvature_defects",
    "coinciding_defects",
    # "num_beak_defects",
    "curvature_min",
    # "curvature_std",
    # "curvature_var",
    "curvature_total",
    # "curvature_median",
    # "curvature_iqr",
    "curvature_90th",
    "curvature_defects_length_mean",
    "curvature_defects_length_max",
    "curvature_defects_length_min",
    # "curvature_defects_depth_mean",
    "curvature_defects_depth_max",
    "curvature_defects_depth_min",
    # "height_defects_length_mean",
    "height_defects_length_max",
    "height_defects_length_min",
    # "height_defects_depth_mean",
    "height_defects_depth_max",
    "height_defects_depth_min",
    # "curvature_defects_volume_mean",
    "curvature_defects_volume_max",
    "curvature_defects_volume_min",
    # "height_defects_volume_mean",
    "height_defects_volume_max",
    "height_defects_volume_min",

]
df = df[cols_to_pca]

pipe = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),  # impute missing values with the median of each column
        ("scale", StandardScaler()),  # scale the data to have mean 0 and variance 1
        ("pca", PCA(n_components=2)),  # reduce the data to 2 principal components
    ]
)

# fit the pipeline to the data, scores are the PCA transformed values
scores = pipe.fit_transform(df)  # run the pipeline on the data
pca = pipe.named_steps["pca"]  # get the PCA step from the pipeline

pc_df = pd.DataFrame(scores, columns=["PC1", "PC2"], index=df.index)

# add columns from the original dataframe to the PCA dataframe
pc_df["grain_id"] = original_df["grain_id"]
pc_df["folder"] = original_df["folder"]

# get the loadings - ie the contribution of each original feature to the PCs.
loadings = pd.DataFrame(pca.components_.T, index=df.columns, columns=["PC1", "PC2"])

# sort the loadings by the sum of PC1 and PC2, so that the most important features are at the top and display them
loadings["sum"] = loadings["PC1"].abs() + loadings["PC2"].abs()
loadings = loadings.sort_values(by="sum", ascending=False).drop(columns=["sum"])

print(f"explained variance ratio: {pca.explained_variance_ratio_}")
print(f"loadings: {loadings}")

# for folder in pc_df["folder"].unique():
#     folder_df = pc_df[pc_df["folder"] == folder]
#     plt.scatter(folder_df["PC1"], folder_df["PC2"], label=folder)
# plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
# plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
# plt.legend()
# plt.show()


def confidence_ellipse(ax, x, y, n_std=3.0, facecolor="none", edgecolor="black", **kwargs):
    if len(x) < 2:
        raise ValueError("x and y must have at least 2 points each")
    # calculate the covariance matrix and its eigenvalues and eigenvectors
    covariance = np.cov(x, y)
    # get the eigenvalues and eigenvectors of the covariance matrix
    values, vectors = np.linalg.eigh(covariance)

    # sort the eigenvalues and eigenvectors in descending order
    order = values.argsort()[::-1]
    values = values[order]
    vectors = vectors[:, order]

    # calculate the angle of the ellipse in degrees by taking the arctan of the ratio of the y component to the x
    # component of the first eigenvector (since the eigenvectors are normalized, we can just use the first eigenvector)
    angle = np.degrees(np.arctan2(*vectors[:, 0][::-1]))
    # calculate the width and height of the ellipse based on the eigenvalues and the number of standard deviations
    width, height = 2 * n_std * np.sqrt(values)

    ellipse = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=width,
        height=height,
        angle=angle,
        facecolor=facecolor,
        edgecolor=edgecolor,
        lw=2,
        **kwargs,
    )
    ax.add_patch(ellipse)


figure, ax = plt.subplots(figsize=(10, 8))
for folder_group in folder_groups:
    folders = folder_group[0]
    colour = folder_group[1]
    group_df = pc_df[pc_df["folder"].isin(folders)]
    ax.scatter(group_df["PC1"], group_df["PC2"], label=", ".join(folders), color=colour, alpha=0.7)
    confidence_ellipse(ax, group_df["PC1"], group_df["PC2"], n_std=3.0, edgecolor=colour, alpha=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.legend()
plt.show()

# plot a kde of the volume means for each sample group
fig, ax = plt.subplots(figsize=(10, 8))
for folder_group in folder_groups:
    folders = folder_group[0]
    colour = folder_group[1]
    group_df = original_df[original_df["folder"].isin(folders)]
    group_df["curvature_defects_volume_mean"].plot.kde(ax=ax, label=", ".join(folders), color=colour)
ax.set_xlabel("Curvature Defects Volume Mean (nm^3)")
ax.set_ylabel("Density")
ax.legend()
plt.show()

# find the minimum volume grain
minimum_volume_grain = original_df.loc[original_df["curvature_defects_volume_mean"].idxmin()]
print(f"Minimum volume grain: {minimum_volume_grain['grain_id']} with volume {minimum_volume_grain['curvature_defects_volume_mean']} nm^3")